In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import torch
from pzest.config import load_config
from pzest.dataset.splits import load_splits
from pzest.dataset.dataset import get_dataloader
from pzest.evaluation.inference import predict
from pzest.models.deepz import DeepZ
from pzest.utils import load_checkpoint

## DeepZ Demo

Visualises predicted redshift PDFs alongside galaxy images for a random sample of the held-out test galaxies. Completely self-contained - loads directly from the processed dataset.

Prerequisites: preprocessing must be complete and at least one model checkpoint must exist in `outputs/checkpoints/`.

Load checkpoint and config

In [ ]:
# Change these together - config must match checkpoint
# baseline_best.pt -> baseline_config.yaml
# model1_best.pt -> model1_config.yaml
# model2_best.pt -> model2_config.yaml
# model3_best.pt -> model3_config.yaml (recommended)
CHECKPOINT_NAME = "model3_best.pt"
CONFIG_PATH = "../config/model3_config.yaml"

config = load_config(CONFIG_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config.paths.figures.mkdir(parents=True, exist_ok=True)
print(f"Device: {device}")
print(f"Checkpoint: {CHECKPOINT_NAME}")

Load model

In [ ]:
model = DeepZ(config)
checkpoint = load_checkpoint(
    model=model,
    path=config.paths.checkpoints_dir / CHECKPOINT_NAME,
    device=device,
)
model = model.to(device)
print(f"Loaded {CHECKPOINT_NAME} - epoch {checkpoint['epoch']} (val σ_NMAD: {checkpoint['val_snmad']:.4f})")

Sample test galaxies

In [ ]:
_, _, test_indices = load_splits(config)

rng = np.random.default_rng(config.data.random_seed)
demo_indices = rng.choice(
    test_indices,
    size=config.evaluation.num_vis_samples,
    replace=False,
)

with h5py.File(config.paths.processed_hdf5_file, "r") as f:
    demo_images = f["images"][np.sort(demo_indices)]
    demo_mags = f["magnitudes"][np.sort(demo_indices)]
    demo_z_true = f["redshift"][np.sort(demo_indices)]

Run inference

In [ ]:
bin_edges = np.linspace(
    config.model.redshift_min,
    config.model.redshift_max,
    config.model.num_bins + 1,
)
bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])

demo_loader = get_dataloader(
    config.paths.processed_hdf5_file,
    np.sort(demo_indices),
    bin_edges,
    config,
    shuffle=False,
)

pdfs, point_estimates = predict(model, demo_loader, bin_edges, device)

Visualise

In [ ]:
n     = config.evaluation.num_vis_samples
fig, axes = plt.subplots(n, 2, figsize=(8, n * 2.5))

for i in range(n):
    axes[i, 0].imshow(demo_images[i, 2], origin="lower", cmap="viridis")
    axes[i, 0].set_title(f"i band | z_true={demo_z_true[i]:.3f}")
    axes[i, 0].axis("off")

    axes[i, 1].plot(bin_centres, pdfs[i], lw=1.5)
    axes[i, 1].axvline(
        demo_z_true[i],
        color="r", lw=1.5, linestyle="--",
        label=f"z_true={demo_z_true[i]:.3f}",
    )
    axes[i, 1].axvline(
        point_estimates[i],
        color="g", lw=1.5, linestyle=":",
        label=f"z_pred={point_estimates[i]:.3f}",
    )
    axes[i, 1].set_xlabel("Redshift")
    axes[i, 1].set_ylabel("P(z)")
    axes[i, 1].legend(fontsize=7)

plt.suptitle(f"DeepZ ({CHECKPOINT_NAME.replace('_best.pt', '')}) — "
             f"redshift PDFs for {n} test galaxies", y=1.01)
plt.tight_layout()
plt.savefig(
    config.paths.figures / f"demo_{CHECKPOINT_NAME.replace('_best.pt', '')}.png",
    dpi=150, bbox_inches="tight"
)
plt.show()